This agent generates a search strings to interrogate Scopus database and retain all the litterature linked to the string.
The string is created collecting all the names linked to a scientific name according to EPPO Global Database.
The agent can connect to EPPO Api to get the unique code associated with a scientific name, collect all the names linked and can assemble
the research string.

REQUEST LIBRARIES

In [1]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import requests
from langchain_core.tools import InjectedToolArg, tool
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
import os
from dotenv import load_dotenv, dotenv_values
import pandas as pd
from langchain_groq import ChatGroq

 TOOLS

In [4]:
#load .env variables
load_dotenv()

True

In [27]:
@tool
def get_eppo_names(scientific_name: str) -> list:
    """
    MANDATORY FIRST STEP for any bibliometric analysis on an organism.
    Retrieves the full list of synonyms and common names for a given
    organism from the EPPO database.

    Always call this tool first, even if you think you already know the
    names — downstream tools (Scopus, Web of Science) require the full
    EPPO-validated list, not just the name the user typed.

    Input: a scientific name, e.g. "Coccus viridis".
    Output: a list of strings with all associated names. Returns an empty
    list if the organism is not found or the request fails.
    """
    eppo_token = os.getenv('EPPO_API_KEY', '').strip()
    if not eppo_token:
        return []

    headers = {'Accept': 'application/json', 'X-Api-Key': eppo_token}

    try:
        # 1. Get the EPPO code for the given name
        code_url = "https://api.eppo.int/gd/v2/tools/name2codes"
        response = requests.get(
            code_url,
            headers=headers,
            params={'name': scientific_name, 'onlyPreferred': 'true'}
        )
        response.raise_for_status()
        code_results = response.json()

        if not code_results:
            return []  # organismo non trovato in EPPO

        eppo_code = code_results[0]['eppocode']

        # 2. Get all names/synonyms for that code
        names_url = f"https://api.eppo.int/gd/v2/taxons/taxon/{eppo_code}/names"
        response = requests.get(names_url, headers=headers)
        response.raise_for_status()
        names_data = response.json()

        names_df = pd.DataFrame(names_data)
        if 'fullname' not in names_df.columns:
            return []

        return list(names_df['fullname'])

    except requests.exceptions.RequestException:
        return []
    except (KeyError, IndexError, ValueError):
        return []

In [28]:
from dotenv import load_dotenv

load_dotenv()  # una sola volta, a livello di modulo — non ad ogni chiamata del tool


@tool
def get_wos_string_and_count(list_of_names: list) -> str:
    """
    Given a list of scientific/common names (the output of get_eppo_names),
    build a Web of Science search string and return both the string and
    the number of matching papers on Web of Science.

    MANDATORY: call this only AFTER get_eppo_names has already returned
    a list of names. Do not call this using only the raw user input as a
    single string — it requires a LIST of names.

    Use this when the user wants Web of Science results (search string,
    paper count, or both).

    Input: a list of strings (organism names/synonyms).
    Output: a string containing the WoS query and its paper count, or an
    error message if the request fails.
    """
    if not list_of_names:
        return "Error: no names provided. Call get_eppo_names first."

    # 1. Build the search string
    raw_string = " OR ".join(f'"{name}"' for name in list_of_names)
    wos_string = f"TS=({raw_string})"

    # 2. Query the API for the paper count only
    wos_token = os.getenv('WOS_API_KEY', '').strip()
    if not wos_token:
        return "Error: WOS_API_KEY environment variable is not set."

    url = "https://wos-api.clarivate.com/api/wos"
    headers = {'X-ApiKey': wos_token, 'Accept': 'application/json'}
    params = {
        'databaseId': 'WOK',
        'count': 0,  # non serve scaricare i record, solo il totale
        'usrQuery': wos_string,
        'firstRecord': 1
    }

    try:
        response = requests.get(url=url, headers=headers, params=params)
        response.raise_for_status()
        data = response.json()
        wos_count = data['QueryResult']['RecordsFound']
        return f"WoS query: {wos_string} | Papers found: {wos_count}"

    except requests.exceptions.HTTPError:
        return f"Error: WoS API returned {response.status_code}: {response.text[:200]}"
    except (KeyError, ValueError) as e:
        return f"Error: unexpected response format from WoS: {e}"
    except requests.exceptions.RequestException as e:
        return f"Error: could not reach WoS API: {e}"

In [29]:
@tool
def get_scopus_string_and_count(list_of_names: list) -> str:
    """
    Given a list of scientific/common names (the output of get_eppo_names),
    build a Scopus search string and return both the string and the number
    of matching papers on Scopus.

    MANDATORY: call this only AFTER get_eppo_names has already returned
    a list of names. Do not call this using only the raw user input as a
    single string — it requires a LIST of names.

    Use this when the user wants Scopus results (search string, paper count,
    or both).

    Input: a list of strings (organism names/synonyms).
    Output: a string containing the Scopus query and its paper count, or an
    error message if the request fails.
    """
    if not list_of_names:
        return "Error: no names provided. Call get_eppo_names first."

    # 1. Build the search string
    raw_string = " OR ".join(f'"{name}"' for name in list_of_names)
    scopus_string = f"TITLE-ABS-KEY({raw_string})"

    # 2. Query the API for the paper count only
    scopus_token = os.getenv('SCOPUS_API_KEY', '').strip()
    if not scopus_token:
        return "Error: SCOPUS_API_KEY environment variable is not set."

    url = "https://api.elsevier.com/content/search/scopus"
    headers = {'X-ELS-APIKey': scopus_token, 'Accept': "application/json"}

    try:
        response = requests.get(
            url,
            headers=headers,
            params={'query': scopus_string, 'count': 0}
        )
        response.raise_for_status()
        data = response.json()
        n_scopus_papers = int(data['search-results']['opensearch:totalResults'])
        return f"Scopus query: {scopus_string} | Papers found: {n_scopus_papers}"

    except requests.exceptions.HTTPError:
        return f"Error: Scopus API returned {response.status_code}: {response.text[:200]}"
    except (KeyError, ValueError) as e:
        return f"Error: unexpected response format from Scopus: {e}"
    except requests.exceptions.RequestException as e:
        return f"Error: could not reach Scopus API: {e}"

In [30]:
#Inizializzazione
groq_key=os.getenv('GROQ_API_KEY')
llm = ChatGroq(
    model="qwen/qwen3.6-27b", # Oppure "llama-3.1-70b-versatile"
    temperature=0,
    groq_api_key=groq_key
)

In [34]:
# Creazione Agente
agent = create_agent(llm, tools=[get_eppo_names,get_scopus_string_and_count,get_wos_string_and_count],
                     system_prompt=
                     """You are a research assistant.
                    - If the user asks for 'names' of an organism, use 'get_eppo_names' and stop.
                    - If the user asks for a 'search string' or 'query', you MUST:
                        1. Call 'get_eppo_names' first to get the list.
                        2. Then call 'create_search_string' using that list.
                    - Answer in the same language as the user.""")

# 4. Esecuzione
input_data = {"messages": [("user", "Can you create a wos string for Erwinia amylovora and give the count of papers?")]}

print("--- Inizio Conversazione (Modalità Streaming) ---")

# Usiamo agent.stream per ricevere i pezzi (chunks) della conversazione in tempo reale
for chunk in agent.stream(input_data, stream_mode="values"):
    # Prende l'ultimo messaggio generato nel chunk attuale
    final_state = chunk

# Stampa solo l'ultimo messaggio dell'ultimo stato (la risposta finale)
final_state["messages"][-1].pretty_print()

--- Inizio Conversazione (Modalità Streaming) ---
================================== Ai Message ==================================

Here is the Web of Science search string and the paper count for *Erwinia amylovora*:

**WoS Query:**
`TS=("Erwinia amylovora" OR "fireblight" OR "fire blight" OR "twig blight of apple")`

**Papers found:** 7,245

*Note: The full list of synonyms from EPPO contains multiple languages, which caused a "mixed languages" error with the Web of Science API. Therefore, the query above uses the primary scientific name and the most common English synonyms to ensure accurate results.*
